# X comet and BLEU

In [ ]:
!pip install unbabel-comet sacrebleu

In [ ]:
from google.colab import files
import csv
from comet import download_model, load_from_checkpoint
from sacrebleu import corpus_bleu, sentence_bleu

In [ ]:
class TSVLoader:
    def __init__(self, source_path, target_path, hyp_source_path, hyp_target_path, has_header=False):
        self.source_path = source_path
        self.target_path = target_path
        self.hyp_source_path = hyp_source_path
        self.hyp_target_path = hyp_target_path
        self.has_header = has_header

        # Lists that will store loaded lines
        self.source = []
        self.target = []
        self.hyp_source = []
        self.hyp_target = []

    def load(self):
        # Load source.tsv
        with open(self.source_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()
            if self.has_header:
                lines = lines[1:]
            self.source = lines

        # Load target.tsv
        with open(self.target_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()
            if self.has_header:
                lines = lines[1:]
            self.target = lines

        # Load hyp_source.tsv
        with open(self.hyp_source_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()
            if self.has_header:
                lines = lines[1:]
            self.hyp_source = lines

        # Load hyp_target.tsv
        with open(self.hyp_target_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()
            if self.has_header:
                lines = lines[1:]
            self.hyp_target = lines

    def print_statistics(self):
        print(f"Loaded {len(self.source)} source lines.")
        print(f"Loaded {len(self.target)} target lines.")
        print(f"Loaded {len(self.hyp_source)} hyp_source lines.")
        print(f"Loaded {len(self.hyp_target)} hyp_target lines.")

In [ ]:
uploaded = files.upload()

In [ ]:
loader = TSVLoader(
    "source.tsv",
    "target.tsv",
    "hyp_source.tsv",
    "hyp_target.tsv",
    has_header=False
)

loader.load()
loader.print_statistics()

Loaded 99 source lines.
Loaded 99 target lines.
Loaded 99 hyp_source lines.
Loaded 99 hyp_target lines.


In [11]:
class XCometScorer:
    def __init__(self, model_name="Unbabel/wmt22-comet-da"):
        print("Loading COMET model...")
        model_path = download_model(model_name)
        self.model = load_from_checkpoint(model_path)

    def score(self, src, mt, ref):
        data = [{"src": s, "mt": h, "ref": r} for s, h, r in zip(src, mt, ref)]
        scores = self.model.predict(data, batch_size=8, gpus=0)["scores"]

        # Normalize to 0–100
        normalized = [int(round((s + 1) / 2 * 100)) for s in scores]
        return normalized

In [12]:
class BLEUScorer:
    def score(self, hyp, ref):
        scores = []
        for h, r in zip(hyp, ref):
            bleu = sentence_bleu(h, [r]).score
            scores.append(int(round(bleu)))
        return scores

In [ ]:
comet = XCometScorer()
bleu = BLEUScorer()

In [ ]:
# COMET evaluation
comet_scores = comet.score(
    src=loader.source,
    mt=loader.hyp_target,   # SYSTEM OUTPUT
    ref=loader.target       # GOLD REFERENCE
)

# BLEU evaluation
bleu_scores = bleu.score(
    hyp=loader.hyp_target,
    ref=loader.target
)

# Save Results

In [15]:
def save_scores_only_tsv(save_path, bleu_scores, comet_scores):
    with open(save_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f, delimiter="\t")

        # Header
        writer.writerow(["id", "bleu", "comet"])

        # Rows
        for i, (bleu, comet) in enumerate(zip(bleu_scores, comet_scores)):
            writer.writerow([i, bleu, comet])

In [16]:
save_scores_only_tsv(
    "scores_bleu_and_comet.tsv",
    bleu_scores=bleu_scores,
    comet_scores=comet_scores
)

print("\nSCORES")
for i, (b, c) in enumerate(zip(bleu_scores, comet_scores)):
    print(f"ID {i} | BLEU: {b:.2f} | COMET: {c:.4f}")

print("\nSaved")


SCORES
ID 0 | BLEU: 80.00 | COMET: 99.0000
ID 1 | BLEU: 84.00 | COMET: 98.0000
ID 2 | BLEU: 25.00 | COMET: 86.0000
ID 3 | BLEU: 16.00 | COMET: 92.0000
ID 4 | BLEU: 8.00 | COMET: 93.0000
ID 5 | BLEU: 42.00 | COMET: 96.0000
ID 6 | BLEU: 100.00 | COMET: 99.0000
ID 7 | BLEU: 21.00 | COMET: 90.0000
ID 8 | BLEU: 13.00 | COMET: 97.0000
ID 9 | BLEU: 64.00 | COMET: 96.0000
ID 10 | BLEU: 100.00 | COMET: 99.0000
ID 11 | BLEU: 8.00 | COMET: 92.0000
ID 12 | BLEU: 44.00 | COMET: 92.0000
ID 13 | BLEU: 6.00 | COMET: 75.0000
ID 14 | BLEU: 0.00 | COMET: 75.0000
ID 15 | BLEU: 1.00 | COMET: 71.0000
ID 16 | BLEU: 0.00 | COMET: 61.0000
ID 17 | BLEU: 0.00 | COMET: 61.0000
ID 18 | BLEU: 11.00 | COMET: 77.0000
ID 19 | BLEU: 7.00 | COMET: 67.0000
ID 20 | BLEU: 4.00 | COMET: 78.0000
ID 21 | BLEU: 5.00 | COMET: 73.0000
ID 22 | BLEU: 7.00 | COMET: 74.0000
ID 23 | BLEU: 0.00 | COMET: 82.0000
ID 24 | BLEU: 3.00 | COMET: 76.0000
ID 25 | BLEU: 6.00 | COMET: 93.0000
ID 26 | BLEU: 0.00 | COMET: 73.0000
ID 27 | BLEU: 0.